# 02 — Experimentos de RAG: embeddings locais, ChromaDB e recuperação

**Objetivo:** experimentar o pipeline de recuperação de políticas da Empório da Música sem utilizar uma chave de API da OpenAI.

Este notebook utiliza um modelo multilíngue local do Sentence Transformers para gerar embeddings. O conteúdo e a documentação do notebook estão em português, assim como **todas as consultas de recuperação**, alinhadas ao contexto de atendimento da loja.

**Fluxo de trabalho**
1. Carregar o PDF de políticas.
2. Extrair o texto preservando os metadados das páginas.
3. Experimentar tamanhos de fragmentos (*chunks*) e sobreposição.
4. Gerar embeddings multilíngues localmente.
5. Criar e persistir uma coleção no ChromaDB.
6. Executar recuperação semântica com consultas em português.
7. Avaliar a relevância dos resultados recuperados e documentar as decisões de engenharia.

> Modelo: `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` (embeddings de 384 dimensões). O modelo é baixado no primeiro uso e passa a ser executado localmente. Não é necessária chave da API da OpenAI nem serviço pago de geração de embeddings.

## 0. Instalação das dependências

In [1]:
%pip -q install -U pypdf sentence-transformers chromadb pandas


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


## 1. Envio do PDF de políticas

In [2]:
from pathlib import Path
from google.colab import files
import re
import pandas as pd

print("Selecione o PDF de políticas da loja:")
uploaded = files.upload()

pdf_files = [name for name in uploaded if name.lower().endswith(".pdf")]
if not pdf_files:
    raise FileNotFoundError("Nenhum arquivo PDF foi enviado.")

PDF_PATH = Path(pdf_files[0])
CHROMA_PATH = Path("/content/chroma_store_policies")
CHROMA_PATH.mkdir(parents=True, exist_ok=True)
COLLECTION_NAME = "store_policies_multilingual_experiment"

print("PDF selecionado:", PDF_PATH.name)
print("Diretório do ChromaDB:", CHROMA_PATH)


/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Selecione o PDF de políticas da loja:


Saving políticas_da_loja.pdf to políticas_da_loja.pdf
PDF selecionado: políticas_da_loja.pdf
Diretório do ChromaDB: /content/chroma_store_policies


## 2. Extração do texto do PDF

In [3]:
from pypdf import PdfReader

reader = PdfReader(str(PDF_PATH))
pages = []

for page_number, page in enumerate(reader.pages, start=1):
    page_text = (page.extract_text() or "").strip()
    if page_text:
        pages.append({
            "text": page_text,
            "page": page_number,
            "source": PDF_PATH.name
        })

if not pages:
    raise ValueError("Não foi possível extrair texto do PDF enviado.")

print(f"Páginas com texto extraído: {len(pages)}")
display(pd.DataFrame([
    {"page": p["page"], "characters": len(p["text"]), "preview": p["text"][:250]}
    for p in pages
]))


Páginas com texto extraído: 8


,page,characters,preview
0,1,397,EMPÓRIO\n \nDA\n \nMÚSICA\n \nSua\n \nmúsica\n...
1,2,2205,Empório\n \nda\n \nMúsica\n \nManual\n \nde\n ...
2,3,1077,Empório\n \nda\n \nMúsica\n \nManual\n \nde\n ...
3,4,2667,Empório\n \nda\n \nMúsica\n \nManual\n \nde\n ...
4,5,1457,Empório\n \nda\n \nMúsica\n \nManual\n \nde\n ...
5,6,2954,Empório\n \nda\n \nMúsica\n \nManual\n \nde\n ...
6,7,2753,Empório\n \nda\n \nMúsica\n \nManual\n \nde\n ...
7,8,1420,Empório\n \nda\n \nMúsica\n \nManual\n \nde\n ...


## 3. Fragmentação (*chunking*): experimento com tamanho e sobreposição

In [4]:
def chunk_pages(pages, chunk_size=900, overlap=150):
    if chunk_size <= 0 or overlap < 0 or overlap >= chunk_size:
        raise ValueError("Utilize chunk_size > overlap >= 0.")

    chunks = []
    for page in pages:
        text = re.sub(r"\s+", " ", page["text"]).strip()
        start = 0

        while start < len(text):
            end = min(start + chunk_size, len(text))

            if end < len(text):
                boundary = max(
                    text.rfind(". ", start, end),
                    text.rfind("; ", start, end),
                    text.rfind(" ", start, end)
                )
                if boundary > start + int(chunk_size * 0.6):
                    end = boundary + 1

            chunk_text = text[start:end].strip()
            if chunk_text:
                chunks.append({
                    "id": f'{page["source"]}-p{page["page"]}-c{len(chunks) + 1}',
                    "text": chunk_text,
                    "metadata": {
                        "source": page["source"],
                        "page": int(page["page"])
                    }
                })

            if end >= len(text):
                break
            start = max(start + 1, end - overlap)

    return chunks


CHUNK_SIZE = 900
OVERLAP = 150

chunks = chunk_pages(pages, chunk_size=CHUNK_SIZE, overlap=OVERLAP)

chunk_df = pd.DataFrame([
    {
        "id": chunk["id"],
        "page": chunk["metadata"]["page"],
        "characters": len(chunk["text"]),
        "text": chunk["text"]
    }
    for chunk in chunks
])

print("Total de fragmentos:", len(chunks))
display(chunk_df.head(10))


Total de fragmentos: 18


,id,page,characters,text
0,políticas_da_loja.pdf-p1-c1,1,297,EMPÓRIO DA MÚSICA Sua música começa aqui. MANU...
1,políticas_da_loja.pdf-p2-c2,2,894,Empório da Música Manual de Políticas e Proced...
2,políticas_da_loja.pdf-p2-c3,2,897,umentos musicais de qualidade na região Centro...
3,políticas_da_loja.pdf-p2-c4,2,223,Domingo e Feriados Fechado Em datas especiais ...
4,políticas_da_loja.pdf-p3-c5,3,801,Empório da Música Manual de Políticas e Proced...
5,políticas_da_loja.pdf-p4-c6,4,899,Empório da Música Manual de Políticas e Proced...
6,políticas_da_loja.pdf-p4-c7,4,897,cados em até 30 (trinta) dias corridos após a ...
7,políticas_da_loja.pdf-p4-c8,4,507,dquiridos em promoções de liquidação com aviso...
8,políticas_da_loja.pdf-p5-c9,5,896,Empório da Música Manual de Políticas e Proced...
9,políticas_da_loja.pdf-p5-c10,5,358,entrar em contato imediatamente com a loja. 5....


In [5]:
# Compare diferentes configurações de fragmentação.
chunk_experiments = []

for size, overlap in [(500, 80), (900, 150), (1200, 200)]:
    experiment_chunks = chunk_pages(
        pages,
        chunk_size=size,
        overlap=overlap
    )
    lengths = [len(chunk["text"]) for chunk in experiment_chunks]

    chunk_experiments.append({
        "chunk_size": size,
        "overlap": overlap,
        "number_of_chunks": len(experiment_chunks),
        "average_characters": round(sum(lengths) / len(lengths), 1),
        "minimum_characters": min(lengths),
        "maximum_characters": max(lengths)
    })

display(pd.DataFrame(chunk_experiments))


,chunk_size,overlap,number_of_chunks,average_characters,minimum_characters,maximum_characters
0,500,80,29,450.3,193,499
1,900,150,18,716.2,223,899
2,1200,200,13,953.6,285,1199


## 4. Geração local de embeddings multilíngues (sem chave de API)

O manual de políticas está escrito em português e a loja atende clientes brasileiros. Portanto, um modelo de embeddings multilíngue é mais adequado do que um modelo focado em inglês para consultas em português.

In [6]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# O modelo é baixado no primeiro uso. Nenhuma chave de API é necessária.
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

document_texts = [chunk["text"] for chunk in chunks]

document_embeddings = embedding_model.encode(
    document_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Modelo de embeddings:", EMBEDDING_MODEL_NAME)
print("Quantidade de embeddings:", len(document_embeddings))
print("Dimensões dos embeddings:", document_embeddings.shape)
print("Amostra do primeiro vetor:", document_embeddings[0][:8])


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo de embeddings: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Quantidade de embeddings: 18
Dimensões dos embeddings: (18, 384)
Amostra do primeiro vetor: [ 0.07008833 -0.03141922 -0.03665372  0.02126042 -0.03228508  0.01241734
 -0.02347486  0.0589514 ]


## 5. Criação e persistência do banco vetorial no ChromaDB

In [7]:
import chromadb

client = chromadb.PersistentClient(path=str(CHROMA_PATH))

# Utilize uma coleção separada para os embeddings locais.
collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)

# A operação upsert permite executar novamente esta célula com segurança para IDs de fragmentos idênticos.
collection.upsert(
    ids=[chunk["id"] for chunk in chunks],
    documents=[chunk["text"] for chunk in chunks],
    metadatas=[chunk["metadata"] for chunk in chunks],
    embeddings=document_embeddings.tolist()
)

print("Coleção:", collection.name)
print("Vetores armazenados:", collection.count())
print("Persistido em:", CHROMA_PATH)


Coleção: store_policies_multilingual_experiment
Vetores armazenados: 18
Persistido em: /content/chroma_store_policies


## 6. Recuperação semântica (top-k)

In [8]:
def retrieve(query, top_k=4):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0].tolist()

    result = collection.query(
        query_embeddings=[query_embedding],
        n_results=min(max(1, top_k), collection.count())
    )

    rows = []
    for index, document in enumerate(result["documents"][0]):
        rows.append({
            "rank": index + 1,
            "page": result["metadatas"][0][index].get("page"),
            "distance": result["distances"][0][index],
            "text": document
        })

    return pd.DataFrame(rows)


query = "Qual é o prazo de retorno para uma reclamação?"
display(retrieve(query, top_k=4))


,rank,page,distance,text
0,1,7,0.422282,erificar se ela encontra-se ativa no sistema. ...
1,2,4,0.586760,cados em até 30 (trinta) dias corridos após a ...
2,3,2,0.640140,Domingo e Feriados Fechado Em datas especiais ...
3,4,4,0.642987,dquiridos em promoções de liquidação com aviso...


In [9]:
# Estas consultas estão intencionalmente em português brasileiro.
# Elas representam perguntas realistas de clientes da loja.
evaluation_queries = [
    "Qual é o prazo de retorno para uma reclamação?",
    "A loja vende acessórios para instrumentos musicais?",
    "O desconto de 5% no PIX pode ser acumulado com uma promoção?",
    "Quais são as condições para troca por preferência?",
    "Quais são os canais de contato da loja?"
]

for query in evaluation_queries:
    print("\\n" + "=" * 100)
    print("CONSULTA:", query)
    display(retrieve(query, top_k=3))


\n====================================================================================================
CONSULTA: Qual é o prazo de retorno para uma reclamação?


,rank,page,distance,text
0,1,7,0.422282,erificar se ela encontra-se ativa no sistema. ...
1,2,4,0.586760,cados em até 30 (trinta) dias corridos após a ...
2,3,2,0.640140,Domingo e Feriados Fechado Em datas especiais ...


\n====================================================================================================
CONSULTA: A loja vende acessórios para instrumentos musicais?


,rank,page,distance,text
0,1,2,0.282826,Empório da Música Manual de Políticas e Proced...
1,2,4,0.413115,Empório da Música Manual de Políticas e Proced...
2,3,2,0.424356,umentos musicais de qualidade na região Centro...


\n====================================================================================================
CONSULTA: O desconto de 5% no PIX pode ser acumulado com uma promoção?


,rank,page,distance,text
0,1,7,0.425909,erificar se ela encontra-se ativa no sistema. ...
1,2,4,0.544315,dquiridos em promoções de liquidação com aviso...
2,3,6,0.579314,disponibilidade de estoque. Produtos esgotados...


\n====================================================================================================
CONSULTA: Quais são as condições para troca por preferência?


,rank,page,distance,text
0,1,7,0.573722,Empório da Música Manual de Políticas e Proced...
1,2,7,0.575927,erificar se ela encontra-se ativa no sistema. ...
2,3,3,0.606116,Empório da Música Manual de Políticas e Proced...


\n====================================================================================================
CONSULTA: Quais são os canais de contato da loja?


,rank,page,distance,text
0,1,6,0.590524,disponibilidade de estoque. Produtos esgotados...
1,2,6,0.638773,durante o horário de funcionamento. Fora do ex...
2,3,8,0.650450,Empório da Música Manual de Políticas e Proced...


## 7. Avaliação manual da recuperação

In [10]:
# Julgamentos manuais de relevância com base nos fragmentos recuperados.
# Um resultado é relevante somente quando seu fragmento contém evidências que respondem à consulta.
manual_evaluation = pd.DataFrame([
    {
        "query": evaluation_queries[0],
        "relevant_at_rank_1": True,
        "relevant_in_top_3": True,
        "relevant_pages": "7",
        "notes": "Rank 1 is the page-7 chunk that contains the complaint escalation rule and the 24-business-hour response deadline."
    },
    {
        "query": evaluation_queries[1],
        "relevant_at_rank_1": True,
        "relevant_in_top_3": True,
        "relevant_pages": "2",
        "notes": "Rank 1 is the page-2 chunk describing the store's exclusive focus on musical instruments and explicitly stating that accessories are not sold."
    },
    {
        "query": evaluation_queries[2],
        "relevant_at_rank_1": False,
        "relevant_in_top_3": True,
        "relevant_pages": "6",
        "notes": "The relevant non-cumulative PIX/promotion rule is in the page-6 chunk at rank 3, not rank 1."
    },
    {
        "query": evaluation_queries[3],
        "relevant_at_rank_1": False,
        "relevant_in_top_3": False,
        "relevant_pages": "",
        "notes": "The top-3 results (pages 7, 7, and 3) do not contain the preference-exchange conditions. The relevant policy is on PDF page 4 and was not retrieved."
    },
    {
        "query": evaluation_queries[4],
        "relevant_at_rank_1": True,
        "relevant_in_top_3": True,
        "relevant_pages": "6",
        "notes": "Rank 1 is a page-6 chunk that contains the primary WhatsApp contact number (67) 3321-4500."
    }
])

display(manual_evaluation)

,query,relevant_at_rank_1,relevant_in_top_3,relevant_pages,notes
0,Qual é o prazo de retorno para uma reclamação?,True,True,7,Rank 1 is the page-7 chunk that contains the c...
1,A loja vende acessórios para instrumentos musi...,True,True,2,Rank 1 is the page-2 chunk describing the stor...
2,O desconto de 5% no PIX pode ser acumulado com...,False,True,6,The relevant non-cumulative PIX/promotion rule...
3,Quais são as condições para troca por preferên...,False,False,,"The top-3 results (pages 7, 7, and 3) do not c..."
4,Quais são os canais de contato da loja?,True,True,6,Rank 1 is a page-6 chunk that contains the pri...


## 8. Conclusões e decisões de engenharia

### 8.1 Configuração executada

- Modelo de embeddings: `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`.
- Embeddings gerados: **18**.
- Dimensionalidade dos vetores: **384**.
- Coleção do ChromaDB: `store_policies_multilingual_experiment`.
- Vetores armazenados: **18**.
- Métrica de recuperação: distância cosseno.
- Conjunto de avaliação: cinco perguntas de clientes escritas em português brasileiro.
- Nenhuma API paga de embeddings foi utilizada; os embeddings foram gerados localmente após o download do modelo.

### 8.2 Efeito da normalização do texto e da nova fragmentação

A célula de fragmentação agora aplica normalização de espaços em branco usando `re.sub(r"\s+", " ", page["text"]).strip()` antes de dividir as páginas. Isso remove as quebras de linha e os espaços repetidos que apareciam na prévia da extração bruta do PDF.

Com a configuração selecionada (900 caracteres, sobreposição de 150), a execução mais recente produziu:
- **18 fragmentos**, em comparação com 22 na execução registrada anteriormente.
- Comprimento médio dos fragmentos: **716,2 caracteres**.
- Comprimento mínimo: **223 caracteres**.
- Comprimento máximo: **899 caracteres**.

O experimento com tamanhos de fragmentos produziu:

| Tamanho do fragmento | Sobreposição | Número de fragmentos | Média de caracteres |
|---:|---:|---:|---:|
| 500 | 80 | 29 | 450,3 |
| 900 | 150 | 18 | 716,2 |
| 1.200 | 200 | 13 | 953,6 |

Esses números descrevem a segmentação, não a qualidade da recuperação. A configuração 900/150 continua sendo um compromisso prático para este experimento, mas não foi comprovada como ideal em comparação com as demais configurações.

A normalização melhorou o texto fornecido à etapa de fragmentação, mas as prévias ainda mostram alguns fragmentos começando ou terminando no meio de palavras (por exemplo, `umentos` e `dquiridos`). A posição inicial da sobreposição é baseada em deslocamentos de caracteres e pode cair dentro de uma palavra. Uma melhoria futura seria mover os limites dos fragmentos para espaços em branco ou limites de frases, tanto no início quanto no fim.

### 8.3 Resultados de recuperação para o conjunto de avaliação em português

Os julgamentos de relevância abaixo foram feitos por meio da inspeção dos fragmentos recuperados, e não apenas de seus números de página ou valores de distância.

| Consulta | Posição 1 relevante? | Relevante no top-3? | Trecho relevante |
|---|---:|---:|---|
| Qual é o prazo de retorno para uma reclamação? | Sim | Sim | Página 7, posição 1 |
| A loja vende acessórios para instrumentos musicais? | Sim | Sim | Página 2, posição 1 |
| O desconto de 5% no PIX pode ser acumulado com uma promoção? | Não | Sim | Página 6, posição 3 |
| Quais são as condições para troca por preferência? | Não | Não | Não recuperado; a política relevante está na página 4 |
| Quais são os canais de contato da loja? | Sim | Sim | Página 6, posição 1 |

Resumo:
- **3/5 (60%)** das consultas retornaram evidência relevante na primeira posição.
- **4/5 (80%)** das consultas retornaram evidência relevante entre os três primeiros resultados.
- A consulta sobre troca por preferência continua sendo uma falha de recuperação.
- A consulta sobre PIX e promoções recupera o trecho correto, mas somente na terceira posição.

### 8.4 A recuperação melhorou em comparação com a execução registrada anteriormente?

Em comparação com a execução anterior do notebook (que utilizou 22 fragmentos e apresentou 2/5 trechos relevantes na primeira posição e 4/5 entre os três primeiros), a execução mais recente mostra:
- A relevância na primeira posição aumentou de **2/5 (40%) para 3/5 (60%)**.
- A relevância entre os três primeiros resultados permaneceu em **4/5 (80%)**.
- A consulta sobre reclamações melhorou significativamente: o primeiro resultado agora é o fragmento da página 7 que contém a regra de encaminhamento da reclamação e o prazo de retorno de 24 horas úteis.
- A consulta sobre acessórios melhorou de um resultado relevante na terceira posição para a primeira posição.
- A consulta sobre canais de contato melhorou de evidência relevante em uma posição inferior para a primeira posição.
- A consulta sobre PIX e promoções passou da primeira para a terceira posição.
- A consulta sobre troca por preferência regrediu: o trecho relevante da página 4 deixou de aparecer entre os três primeiros resultados.

**Conclusão:** houve uma melhoria mensurável na relevância do primeiro resultado, mas não uma melhoria universal. A cobertura dos três primeiros resultados foi mantida, enquanto os casos específicos de falha mudaram. Este é um pequeno conjunto de teste manual com cinco consultas; portanto, esses percentuais são indicadores diagnósticos, não um benchmark estatisticamente robusto.

### 8.5 Análise por consulta e próximas ações

1. **Resposta a reclamações — sucesso na primeira posição.** O fragmento recuperado da página 7 contém a orientação para ouvir com empatia, registrar e encaminhar a reclamação, com retorno em até 24 horas úteis. Esta é uma política importante e deve permanecer nos testes de regressão.
2. **Acessórios — sucesso na primeira posição.** O fragmento da página 2 afirma explicitamente que a loja comercializa exclusivamente instrumentos musicais e não vende acessórios.
3. **PIX e promoções — relevante somente na terceira posição.** O trecho da página 6 informa que as promoções não são cumulativas e que o desconto de 5% no PIX não se aplica a preços já promocionais. Melhorar a classificação para essa consulta, por exemplo, testando uma consulta mais específica e limites alternativos de fragmentação.
4. **Troca por preferência — falha de recuperação.** O trecho da página 4 informa que trocas por preferência são permitidas em até sete dias, sujeitas à disponibilidade do produto, ao perfeito estado e à embalagem original. Esse trecho não aparece entre os três primeiros resultados.